# Pipeline de razonamiento, búsqueda y resultados
**Caso A: minería de asteroides** · Challenge 1 · Machine Learning · UPCH

Flujo: Problema → Espacio de estados → Estados, operadores y restricciones → Exploración → Búsqueda → Resultados.

> **Aviso:** los resultados de este notebook dependen de supuestos didácticos. **No son pruebas científicas** de viabilidad ni de inviabilidad económica.

## 1. Presentación del problema
**Problema:** ¿puede considerarse económicamente viable la minería de asteroides hoy, con la información de la infografía?

**Distinción clave:** la infografía reporta crecimiento *proyectado* del mercado (CAGR 21,4 %, 2026-2030), no rentabilidad de misiones.

**Postura (hipótesis crítica):** la viabilidad actual no está demostrada porque la infografía no reporta costos de extracción, cantidad y concentración de minerales, probabilidad de éxito técnico, tiempo de retorno ni marco regulatorio.

## 2. Representación del espacio de estados
Un estado es una tupla de 5 variables discretas; `None` significa *desconocido*. Los niveles son **abstracciones propias**; los nombres de las variables corresponden a los aspectos *no reportados* de la infografía.

In [ ]:
import itertools
from collections import deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

VARIABLES = {
    "costo_extraccion":  ["bajo", "medio", "alto"],      # rango 0,1,2 (ver RANGO)
    "recurso":           ["bajo", "medio", "alto"],
    "exito_tecnico":     ["bajo", "medio", "alto"],
    "retorno_inversion": ["corto", "medio", "largo"],
    "marco_regulatorio": ["incierto", "claro"],
}
NOMBRES = list(VARIABLES)
RANGO = {"bajo": 0, "medio": 1, "alto": 2}
ESTADO_INICIAL = (None,) * len(NOMBRES)

# Lo que la infografía reporta de cada variable: nada (todas figuran como "no reportadas")
INFOGRAFIA = {v: "no reportado" for v in NOMBRES}
print(pd.Series(INFOGRAFIA, name="Estado según la infografía").to_frame())
print("Estado inicial:", ESTADO_INICIAL)

## 3. Estados, operadores y restricciones
- **Operador:** `asignar(variable, nivel)` sobre una variable desconocida (adoptar un supuesto u obtener un dato).
- **Consistencia (R1, R2):** descartan combinaciones incoherentes.
- **Viabilidad (V1-V6):** definen el estado objetivo.

In [ ]:
def dict_estado(e):
    return {n: v for n, v in zip(NOMBRES, e)}

def consistente(e):
    d = dict_estado(e)
    # R1: éxito técnico bajo => retorno no corto
    if d["exito_tecnico"] == "bajo" and d["retorno_inversion"] == "corto":
        return False
    # R2: marco incierto => retorno no corto
    if d["marco_regulatorio"] == "incierto" and d["retorno_inversion"] == "corto":
        return False
    return True

# Condiciones de viabilidad. Cada una devuelve True/False, o None si aún no se puede evaluar.
def v1(d): return None if d["costo_extraccion"] is None else d["costo_extraccion"] != "alto"
def v2(d): return None if d["recurso"] is None else d["recurso"] != "bajo"
def v3(d):
    if d["recurso"] is None or d["costo_extraccion"] is None: return None
    return RANGO[d["recurso"]] > RANGO[d["costo_extraccion"]]
def v4(d): return None if d["retorno_inversion"] is None else d["retorno_inversion"] != "largo"
def v5(d): return None if d["exito_tecnico"] is None else d["exito_tecnico"] != "bajo"
def v6(d): return None if d["marco_regulatorio"] is None else d["marco_regulatorio"] == "claro"
VIABILIDAD = {"V1 costo no alto": v1, "V2 recurso no bajo": v2, "V3 recurso > costo": v3,
              "V4 retorno no largo": v4, "V5 éxito no bajo": v5, "V6 marco claro": v6}

def viola_viabilidad(e):
    """True si ya se incumple alguna condición V (poda)."""
    d = dict_estado(e)
    return any(f(d) is False for f in VIABILIDAD.values())

def es_objetivo(e):
    return None not in e and consistente(e) and not viola_viabilidad(e)

def sucesores(e, usar_poda=False):
    """Operador asignar(variable, nivel) para cada variable desconocida."""
    for i, v in enumerate(e):
        if v is None:
            for nivel in VARIABLES[NOMBRES[i]]:
                nuevo = e[:i] + (nivel,) + e[i+1:]
                if not consistente(nuevo):
                    continue
                if usar_poda and viola_viabilidad(nuevo):
                    continue
                yield nuevo

total_parciales = int(np.prod([len(l) + 1 for l in VARIABLES.values()]))
total_completos = int(np.prod([len(l) for l in VARIABLES.values()]))
print("Estados posibles (con '?'):", total_parciales, "| estados completos:", total_completos)

## 4. Exploración de escenarios
Se implementan **BFS** (anchura) y **DFS** (profundidad) con control de estados visitados.

**Justificación:** todos los operadores tienen costo 1, así que BFS garantiza el camino más corto al primer escenario viable (mínimo de supuestos necesarios). DFS se incluye como comparación: usa menos memoria pero no garantiza el camino más corto. Como el espacio es pequeño, ambos recorren todo el espacio para enumerar los escenarios viables.

In [ ]:
def buscar(estrategia="bfs", usar_poda=False):
    """Explora todo el espacio alcanzable. Devuelve objetivos, nodos expandidos y datos del primer objetivo."""
    frontera = deque([(ESTADO_INICIAL, 0)])
    visitados = {ESTADO_INICIAL}
    objetivos, expandidos = [], 0
    primero = None
    while frontera:
        e, prof = frontera.popleft() if estrategia == "bfs" else frontera.pop()
        expandidos += 1
        if es_objetivo(e):
            objetivos.append(e)
            if primero is None:
                primero = {"estado": e, "profundidad": prof, "expandidos_hasta_primero": expandidos}
            continue
        for s in sucesores(e, usar_poda):
            if s not in visitados:
                visitados.add(s)
                frontera.append((s, prof + 1))
    return {"objetivos": objetivos, "expandidos": expandidos, "visitados": len(visitados), "primero": primero}

res = {
    "BFS sin poda": buscar("bfs", False),
    "DFS sin poda": buscar("dfs", False),
    "BFS con poda": buscar("bfs", True),
    "DFS con poda": buscar("dfs", True),
}
tabla = pd.DataFrame({
    k: {"estados visitados": r["visitados"], "nodos expandidos": r["expandidos"],
        "escenarios objetivo": len(r["objetivos"]),
        "nodos hasta el 1.er objetivo": r["primero"]["expandidos_hasta_primero"],
        "profundidad del 1.er objetivo": r["primero"]["profundidad"]}
    for k, r in res.items()}).T
tabla

## 5. Búsqueda de escenarios que cumplen las restricciones
Escenarios objetivo (hipotéticos) hallados y qué niveles aparecen en ellos.

In [ ]:
objetivos = res["BFS sin poda"]["objetivos"]
assert set(objetivos) == set(res["DFS con poda"]["objetivos"]), "Las estrategias deberían coincidir"
df_obj = pd.DataFrame(objetivos, columns=NOMBRES)
print("Escenarios objetivo:", len(df_obj), "de", total_completos, "estados completos")
display(df_obj)

# Todos los estados completos consistentes, y la proporción que cumple la viabilidad por nivel
completos = [e for e in itertools.product(*VARIABLES.values()) if consistente(e)]
df_all = pd.DataFrame(completos, columns=NOMBRES)
df_all["viable"] = [es_objetivo(e) for e in completos]
print("Completos consistentes:", len(df_all), "| viables:", int(df_all.viable.sum()),
      f"({df_all.viable.mean():.1%})")

filas = []
for v in NOMBRES:
    for nivel in VARIABLES[v]:
        sub = df_all[df_all[v] == nivel]
        filas.append({"variable": v, "nivel": nivel, "escenarios": len(sub),
                      "viables": int(sub.viable.sum()), "tasa_viable": sub.viable.mean() if len(sub) else 0})
sens = pd.DataFrame(filas)
sens

In [ ]:
os.makedirs("imagenes", exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) tasa de escenarios viables por nivel de cada variable
etiquetas = [f"{r.variable}\n{r.nivel}" for r in sens.itertuples()]
colores = plt.cm.tab10(pd.factorize(sens.variable)[0])
axes[0].bar(range(len(sens)), sens.tasa_viable, color=colores)
axes[0].set_xticks(range(len(sens))); axes[0].set_xticklabels(etiquetas, rotation=90, fontsize=7)
axes[0].set_ylabel("Proporción de escenarios viables")
axes[0].set_title("Condiciones necesarias: niveles con tasa 0 hacen inviable el escenario")

# (b) costo de la búsqueda
axes[1].bar(tabla.index, tabla["nodos expandidos"], color=["#4c72b0", "#dd8452", "#4c72b0", "#dd8452"])
axes[1].set_ylabel("Nodos expandidos"); axes[1].set_title("Costo de exploración por estrategia")
plt.setp(axes[1].get_xticklabels(), rotation=20)
plt.tight_layout()
plt.savefig("imagenes/busqueda_resultados.png", dpi=140, bbox_inches="tight")
plt.show()

## 6. Presentación e interpretación de resultados
Contraste entre condiciones necesarias y lo que informa la infografía.

In [ ]:
cond = pd.DataFrame({
    "condición de viabilidad (supuesto del modelo)": list(VIABILIDAD),
    "¿informada por la infografía?": ["No reportado"] * len(VIABILIDAD),
    "¿verificable con la infografía?": ["No"] * len(VIABILIDAD),
})
display(cond)

d_inicial = dict_estado(ESTADO_INICIAL)
desconocidas = [n for n, v in d_inicial.items() if v is None]
print(f"Variables fijadas por la infografía: {len(NOMBRES) - len(desconocidas)} de {len(NOMBRES)}")
print("Variables desconocidas:", desconocidas)
p = res["BFS sin poda"]["primero"]
print("\nCamino mínimo (BFS) hasta un escenario viable:", p["profundidad"], "asignaciones →", p["estado"])
print("Es decir, hacen falta como mínimo", p["profundidad"], "datos/supuestos que la infografía no aporta.")

### Interpretación
- Existen escenarios **hipotéticos** que cumplen todas las condiciones; por tanto, la viabilidad no queda descartada por el modelo. Para llegar a uno hay que fijar **las 5 variables**, y la infografía no fija ninguna.
- La tasa de viabilidad por nivel muestra qué condiciones son necesarias (por ejemplo, ningún escenario con regulación incierta o costo alto es viable) y cómo cambia el resultado con cada supuesto.
- BFS y DFS hallan el mismo conjunto de escenarios; la poda reduce nodos expandidos sin cambiar el resultado (ver tabla).
- **Postura del proyecto:** como la infografía no permite confirmar ninguna de las condiciones, la viabilidad económica actual **no está demostrada**.

### Limitaciones
- Variables discretas y umbrales elegidos por el autor: los números de escenarios dependen del diseño del modelo.
- No hay montos, probabilidades ni plazos reales; el modelo no calcula rentabilidad.
- Que existan escenarios viables no los hace probables, y que la infografía no informe las variables no prueba inviabilidad: solo impide confirmar la viabilidad con esa fuente.
- Información que faltaría: costos de misión y extracción, cantidad y concentración del recurso, precios y demanda, probabilidad de éxito, plazo de retorno, costo del capital, marco legal y efectos ambientales y geopolíticos.